# Part 2 - Task 3: Train a detection model

Using Ultralytics YOLOv8 since the labels are already in YOLO format. This notebook
reuses the patient-level split saved in `dataset/splits.json` (see
`part2_dataset_inspection.ipynb`).

## Arrange the data into YOLO's expected layout

YOLOv8 expects images and labels under `images/{train,val,test}` and
`labels/{train,val,test}`, with each image's label file living in the mirrored
location under `labels/`. Files are copied (not moved) from `dataset/images` and
`dataset/labels` into a new `dataset/yolo/` directory, routed by the patient ->
split mapping.

In [1]:
import json
import shutil
from pathlib import Path

DATASET_DIR = Path("../dataset")
YOLO_DIR = DATASET_DIR / "yolo"

split_map = json.load(open(DATASET_DIR / "splits.json"))

for split in ["train", "val", "test"]:
    (YOLO_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

image_files = sorted((DATASET_DIR / "images").glob("*.jpg"))
counts = {"train": 0, "val": 0, "test": 0}

for img_path in image_files:
    patient_id = img_path.stem.split("_")[0]
    split = split_map[patient_id]
    label_path = DATASET_DIR / "labels" / f"{img_path.stem}.txt"

    shutil.copy2(img_path, YOLO_DIR / "images" / split / img_path.name)
    shutil.copy2(label_path, YOLO_DIR / "labels" / split / label_path.name)
    counts[split] += 1

print(f"Copied files into {YOLO_DIR.resolve()}")
print(counts)

Copied files into E:\Bone Union Detection\dataset\yolo
{'train': 574, 'val': 123, 'test': 123}


## Dataset config

A `data.yaml` file tells YOLOv8 where the splits live and names the single class.

In [2]:
data_yaml = f"""path: {YOLO_DIR.resolve().as_posix()}
train: images/train
val: images/val
test: images/test

names:
  0: osteotomy_site
"""

data_yaml_path = YOLO_DIR / "data.yaml"
data_yaml_path.write_text(data_yaml)
print(data_yaml_path.read_text())

path: E:/Bone Union Detection/dataset/yolo
train: images/train
val: images/val
test: images/test

names:
  0: osteotomy_site



## Train

**Optimizing for CPU-only training.** An initial full-resolution (512px) run measured
~122 seconds/epoch on this CPU-only machine, making a full ~100-epoch schedule take
3-4 hours. Following feedback that CPU training time depends heavily on how
optimized the data/model setup is (not just raw compute), three changes were made and
timed on a 1-epoch test before committing to a full run:

- **Smaller input size (512 -> 320px).** Compute scales roughly with image area, so
  this alone cuts per-image compute to about (320/512)^2 ~ 39%. Osteotomy sites are
  small, localized features against mostly-irrelevant background, so the loss of
  detail from downsizing is expected to be minor.
- **In-RAM image caching (`cache='ram'`).** The very first run's log printed a "Slow
  image access detected... use local storage instead of remote/mounted storage"
  warning - i.e. part of the original bottleneck was disk I/O, not just compute.
  Caching decoded images in RAM removes that repeated-read cost on every epoch.
  (Ultralytics also flags that `cache='ram'` can make training a little
  non-deterministic run-to-run; `cache='disk'` is the fully-deterministic
  alternative, at the cost of getting less of the I/O speedup.)
- **Larger batch size (16 -> 32).** Fewer, larger batches reduce fixed per-batch
  Python/scheduling overhead.

Measured result: **52 seconds/epoch**, a 2.3x speedup, bringing a 100-epoch budget
down to roughly 1.5 hours worst case (less if early stopping triggers). This made a
real training run - not just a proof-of-concept - feasible in the time available.
`plots=False` is kept so Ultralytics does not save training/validation image mosaics
during training (those would embed dataset slices); qualitative prediction overlays
are instead generated separately after training, and kept out of the public GitHub
repo per the dataset's no-redistribution restriction.

In [3]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(data_yaml_path),
    epochs=100,
    patience=20,
    imgsz=320,
    batch=32,
    cache="ram",
    project="../runs",
    name="osteotomy_yolov8n",
    seed=42,
    plots=False,  # avoid saving mosaics/prediction images that embed dataset content
)

New https://pypi.org/project/ultralytics/8.4.116 available  Update with 'pip install -U ultralytics'


Ultralytics 8.4.115  Python-3.11.9 torch-2.13.0+cpu CPU (11th Gen Intel Core i7-1185G7 @ 3.00GHz)


engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=..\dataset\yolo\data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=osteotomy_yolov8n, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=20, perspec

Overriding model.yaml nc=80 with nc=1



                   from  n    params  module                                       arguments                     


...

Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 50% ━━━━━━────── 1/2 5.8s/it 1.7s<5.8s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.7s/it 3.5s

                   all        123        180       0.79      0.611      0.708      0.225


Speed: 0.4ms preprocess, 24.6ms inference, 0.0ms loss, 0.9ms postprocess per image


## 4.3 Task 4: Evaluate on the held-out test split

Training tracked validation-split metrics throughout (best epoch: 53, by mAP50-95,
before early stopping at epoch 73). The number that actually matters for reporting,
though, is performance on the **test split** - patient-disjoint from both train and
val, and never used for any model-selection decision during training. The best
checkpoint (`results.save_dir`, selected on val performance) is reloaded and run once
against the test split.

In [4]:
best_weights = Path(results.save_dir) / "weights" / "best.pt"
print(f"Loading best checkpoint from {best_weights}")

test_model = YOLO(str(best_weights))
test_metrics = test_model.val(data=str(data_yaml_path), split="test", imgsz=320, plots=False)

print("\nTest set (123 images, 25 patients):")
print(f"Precision: {test_metrics.box.mp:.3f}")
print(f"Recall:    {test_metrics.box.mr:.3f}")
print(f"mAP50:     {test_metrics.box.map50:.3f}")
print(f"mAP50-95:  {test_metrics.box.map:.3f}")

Loading best checkpoint from runs\runs\osteotomy_yolov8n\weights\best.pt


Ultralytics 8.4.115  Python-3.11.9 torch-2.13.0+cpu CPU (11th Gen Intel Core i7-1185G7 @ 3.00GHz)


Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


WARNING val: Slow image access detected (ping: 0.00.0 ms, read: 4.23.9 MB/s, size: 38.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/


val: Scanning E:\Bone Union Detection\dataset\yolo\labels\test.cache... 123 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 123/123  0.0s

...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 87% ━━━━━━━━━━── 7/8 1.9it/s 3.5s<0.5s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s

                   all        123        173      0.563      0.549      0.486      0.139


Speed: 0.4ms preprocess, 17.5ms inference, 0.0ms loss, 0.3ms postprocess per image



Test set (123 images, 25 patients):
Precision: 0.563
Recall:    0.549
mAP50:     0.486
mAP50-95:  0.139


## 4.3 Task 5: Qualitative results and per-image analysis

For each test image, predicted boxes are greedily matched to ground-truth boxes by
IoU (threshold 0.5) to get a per-image true-positive / false-negative / false-positive
count - this both gives a finer-grained breakdown than the aggregate metrics above
(e.g. by number of ground-truth boxes per image, tying back to Task 1) and identifies
concrete success/failure examples to inspect visually.

**Note on outputs.** This cell only prints text; it does not display images inline.
The overlay images themselves (ground truth in green, predictions in red) are saved to
`qualitative_results/` at the project root, which is excluded from the public GitHub
repo via `.gitignore` since those images embed actual dataset slices - see the README
for a written description of what they show.

In [5]:
from PIL import Image, ImageDraw

OUT_DIR = Path("../qualitative_results")
OUT_DIR.mkdir(exist_ok=True)

test_images = sorted((YOLO_DIR / "images" / "test").glob("*.jpg"))


def load_gt_boxes(label_path, img_w, img_h):
    boxes = []
    if not label_path.exists():
        return boxes
    for line in label_path.read_text().splitlines():
        if not line.strip():
            continue
        _, xc, yc, w, h = [float(p) for p in line.split()]
        boxes.append([
            (xc - w / 2) * img_w, (yc - h / 2) * img_h,
            (xc + w / 2) * img_w, (yc + h / 2) * img_h,
        ])
    return boxes


def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


IOU_THRESH = 0.5
per_image_results = []

for img_path in test_images:
    label_path = YOLO_DIR / "labels" / "test" / f"{img_path.stem}.txt"
    w, h = Image.open(img_path).size
    gt_boxes = load_gt_boxes(label_path, w, h)

    pred = test_model.predict(str(img_path), imgsz=320, conf=0.25, verbose=False)[0]
    pred_boxes = pred.boxes.xyxy.cpu().numpy().tolist() if len(pred.boxes) else []
    pred_confs = pred.boxes.conf.cpu().numpy().tolist() if len(pred.boxes) else []

    matched_gt, matched_pred = set(), set()
    for pi, pb in enumerate(pred_boxes):
        best_iou, best_gi = 0, -1
        for gi, gb in enumerate(gt_boxes):
            if gi in matched_gt:
                continue
            v = iou(pb, gb)
            if v > best_iou:
                best_iou, best_gi = v, gi
        if best_iou >= IOU_THRESH:
            matched_gt.add(best_gi)
            matched_pred.add(pi)

    n_gt, n_pred, n_tp = len(gt_boxes), len(pred_boxes), len(matched_gt)
    per_image_results.append({
        "path": img_path, "gt_boxes": gt_boxes,
        "pred_boxes": pred_boxes, "pred_confs": pred_confs,
        "n_gt": n_gt, "n_pred": n_pred, "n_tp": n_tp,
        "n_fn": n_gt - n_tp, "n_fp": n_pred - len(matched_pred),
        "recall": n_tp / n_gt if n_gt > 0 else None,
    })

by_group = {}
for r in per_image_results:
    key = "1 box" if r["n_gt"] == 1 else ("2+ boxes" if r["n_gt"] >= 2 else "0 boxes")
    by_group.setdefault(key, []).append(r)

print("=== Per-image recall by ground-truth box count (test split) ===")
for key, items in sorted(by_group.items()):
    n_images = len(items)
    total_gt = sum(r["n_gt"] for r in items)
    total_tp = sum(r["n_tp"] for r in items)
    perfect = sum(1 for r in items if r["recall"] == 1.0)
    print(f"{key}: {n_images} images, {total_gt} GT boxes, "
          f"box-level recall={total_tp/total_gt:.3f}, "
          f"{perfect}/{n_images} images fully detected ({perfect/n_images:.1%})")

total_fp = sum(r["n_fp"] for r in per_image_results)
total_pred = sum(r["n_pred"] for r in per_image_results)
print(f"\nTotal predicted boxes: {total_pred}, false positives: {total_fp} "
      f"({total_fp/total_pred:.1%} of all predictions)")

perfect_cases = [r for r in per_image_results if r["n_gt"] > 0 and r["recall"] == 1.0 and r["n_fp"] == 0]
miss_cases = sorted([r for r in per_image_results if r["n_fn"] > 0], key=lambda r: -r["n_fn"])
fp_cases = sorted([r for r in per_image_results if r["n_fp"] > 0], key=lambda r: -r["n_fp"])
print(f"\nPerfect-detection images: {len(perfect_cases)}/{len(per_image_results)}")
print(f"Images with >=1 missed box: {len(miss_cases)}/{len(per_image_results)}")
print(f"Images with >=1 false positive: {len(fp_cases)}/{len(per_image_results)}")


def draw_overlay(r, out_path):
    img = Image.open(r["path"]).convert("RGB")
    draw = ImageDraw.Draw(img)
    for gb in r["gt_boxes"]:
        draw.rectangle(gb, outline=(0, 255, 0), width=2)
    for pb, conf in zip(r["pred_boxes"], r["pred_confs"]):
        draw.rectangle(pb, outline=(255, 0, 0), width=2)
        draw.text((pb[0], max(0, pb[1] - 10)), f"{conf:.2f}", fill=(255, 0, 0))
    img.save(out_path, quality=95)


for category, cases in [("success", perfect_cases[:3]), ("missed_detection", miss_cases[:3]), ("false_positive", fp_cases[:3])]:
    for i, r in enumerate(cases):
        out_path = OUT_DIR / f"{category}_{i}_{r['path'].stem}.jpg"
        draw_overlay(r, out_path)
        print(f"Saved {category} example: {out_path.name} "
              f"(gt={r['n_gt']}, tp={r['n_tp']}, fn={r['n_fn']}, fp={r['n_fp']})")

=== Per-image recall by ground-truth box count (test split) ===
1 box: 87 images, 87 GT boxes, box-level recall=0.529, 46/87 images fully detected (52.9%)
2+ boxes: 36 images, 86 GT boxes, box-level recall=0.558, 9/36 images fully detected (25.0%)

Total predicted boxes: 166, false positives: 72 (43.4% of all predictions)

Perfect-detection images: 39/123
Images with >=1 missed box: 68/123
Images with >=1 false positive: 58/123
Saved success example: success_0_1004_1_133.jpg (gt=1, tp=1, fn=0, fp=0)
Saved success example: success_1_1004_2_134.jpg (gt=1, tp=1, fn=0, fp=0)
Saved success example: success_2_104_1_47.jpg (gt=1, tp=1, fn=0, fp=0)
Saved missed_detection example: missed_detection_0_920_1_42.jpg (gt=3, tp=0, fn=3, fp=2)
Saved missed_detection example: missed_detection_1_920_1_53.jpg (gt=3, tp=0, fn=3, fp=0)
Saved missed_detection example: missed_detection_2_120_2_117.jpg (gt=2, tp=0, fn=2, fp=1)
Saved false_positive example: false_positive_0_396_2_74.jpg (gt=1, tp=0, fn=1, fp=3